In [31]:
import os
import typing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import logit as sm_logit
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import roc_auc_score, PrecisionRecallDisplay, f1_score, auc, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.feature_selection import RFECV
import xgboost as xgb
import shap
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind, shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

# Optional libraries
try:
    from upsetplot import UpSet
except ImportError:
    UpSet = None
try:
    from matplotlib_venn import venn2
except ImportError:
    venn2 = None

# Plotting Config
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
color_palette = sns.color_palette("colorblind")

if not os.path.exists('visualizations'):
    os.makedirs('visualizations')

def save_plot(filename, tight_layout=True):
    if tight_layout:
        plt.tight_layout()
    plt.savefig(f"visualizations/{filename}.png", dpi=300)
    print(f"Saved plot: {filename}.png")
    plt.close()

# Modular Functions
def perform_stat_test(group0, group1, col):
    stat_n0, p_n0 = shapiro(group0)
    stat_n1, p_n1 = shapiro(group1)
    if p_n0 > 0.05 and p_n1 > 0.05:
        stat, p = ttest_ind(group0, group1)
        test_type = 't-test'
    else:
        stat, p = mannwhitneyu(group0, group1)
        test_type = 'Mann-Whitney'
    effect = (group1.mean() - group0.mean()) / group0.std()  # Cohen's d
    return test_type, p, effect

def calc_cramers_v(chi2, n, k1, k2):
    return np.sqrt(chi2 / (n * min(k1-1, k2-1)))


In [ ]:

file_path = 'diabetes_binary_health_indicators_BRFSS2015.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' successfully loaded.")
except FileNotFoundError:
    print("="*80)
    print(f"FATAL ERROR: The data file '{file_path}' was not found.")
    print("Please download from: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset")
    print("="*80)
    exit()

# Initial Inspection
print("First 5 rows:\n", df.head())
print("\nData Info:\n")
df.info()
print("\nNull Values:\n", df.isnull().sum())
print("\nUnique Counts:\n", df.nunique())

# Check for constants
constants = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Constant/Near-Constant Columns: {constants}")

# Drop duplicates
print(f"Shape before cleaning: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df.shape}")

# Data Type Optimization
for col in df.columns:
    if df[col].dtype == 'float64':
        df[col] = df[col].astype(int)

# Feature Mappings
binary_cols = ['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
               'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
               'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk']
df['Sex'] = df['Sex'].map({0: 'Female', 1: 'Male'})
genhlth_map = {1: 'Excellent', 2: 'Very Good', 3: 'Good', 4: 'Fair', 5: 'Poor'}
df['GenHlth'] = df['GenHlth'].map(genhlth_map)
age_map = {1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39', 5: '40-44',
           6: '45-49', 7: '50-54', 8: '55-59', 9: '60-64', 10: '65-69',
           11: '70-74', 12: '75-79', 13: '80 or older'}
df['Age'] = df['Age'].map(age_map)
edu_map = {1: 'Never attended/Kindergarten', 2: 'Grades 1-8', 3: 'Grades 9-11',
           4: 'Grade 12/GED', 5: 'College 1-3 years', 6: 'College 4+ years'}
df['Education'] = df['Education'].map(edu_map)
income_map = {1: '<$10k', 2: '$10k-$15k', 3: '$15k-$20k', 4: '$20k-$25k',
              5: '$25k-$35k', 6: '$35k-$50k', 7: '$50k-$75k', 8: '>$75k'}
df['Income'] = df['Income'].map(income_map)

df_viz = df.copy()
for col in binary_cols:
    df_viz[col] = df_viz[col].map({0: 'No', 1: 'Yes'})

# Feature Engineering from 1.py
df['NumComorbidities'] = df[['HighBP', 'HighChol', 'Stroke', 'HeartDiseaseorAttack', 'DiffWalk']].sum(axis=1)
df_viz['NumComorbidities'] = df['NumComorbidities']

# Create Numeric Age for Modeling
age_map_numeric = {'18-24':1, '25-29':2, '30-34':3, '35-39':4, '40-44':5, '45-49':6, '50-54':7, '55-59':8, '60-64':9, '65-69':10, '70-74':11, '75-79':12, '80 or older':13}
if 'Age_numeric' not in df.columns:
    df['Age_numeric'] = df['Age'].map(age_map_numeric).astype(int)
    df_viz['Age_numeric'] = df['Age_numeric']
print("Added 'Age_numeric' column for modeling.")


FATAL ERROR: The data file 'diabetes_binary_health_indicators_BRFSS2015.csv' was not found.
Please download from: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset
First 5 rows:
    Diabetes_binary  HighBP  HighChol  CholCheck  BMI  Smoker  Stroke  \
0                0       1         1          1   40       1       0   
1                0       0         0          0   25       1       0   
2                0       1         1          1   28       0       0   
3                0       1         0          1   27       0       0   
4                0       1         1          1   24       0       0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...     Sex    Age  \
0                     0             0       0  ...  Female  60-64   
1                     0             1       0  ...  Female  50-54   
2                     0             0       1  ...  Female  60-64   
3                     0             1       1  ...  Female  70-74   
4              

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

: 

In [33]:
continuous_features = ['BMI', 'MentHlth', 'PhysHlth']
categorical_features = ['GenHlth', 'Age', 'Education', 'Income'] + binary_cols

# Continuous: Hist/KDE/Box + Stats
for col in continuous_features:
    print(f"\nStats for {col}:\n", df[col].describe())
    print(f"Skewness: {stats.skew(df[col])}, Kurtosis: {stats.kurtosis(df[col])}")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.histplot(data=df, x=col, kde=True, ax=axes[0], color=color_palette[0])
    axes[0].set_title(f'Distribution of {col}')
    sns.boxplot(x=df[col], ax=axes[1], color=color_palette[1])
    axes[1].set_title(f'Boxplot of {col}')
    save_plot(f'2_univariate_{col}_distribution')

# Categorical/Binary: Bar charts + Imbalance
for col in categorical_features:
    proportions = df_viz[col].value_counts(normalize=True).sort_index()
    sns.barplot(x=proportions.index, y=proportions.values, palette='viridis')
    plt.title(f'Percentage Distribution of {col}')
    plt.ylabel('Proportion')
    plt.xlabel(col)
    plt.xticks(rotation=45, ha='right')
    save_plot(f'2_univariate_percent_{col}')
    
    if col in binary_cols:
        ratio = proportions.get('Yes', 0) / proportions.get('No', 0)
        print(f"Imbalance ratio (Yes/No) for {col}: {ratio:.2f}")



Stats for BMI:
 count    229474.000000
mean         28.687507
std           6.789204
min          12.000000
25%          24.000000
50%          27.000000
75%          32.000000
max          98.000000
Name: BMI, dtype: float64
Skewness: 2.0634617823284995, Kurtosis: 10.46657448205856
Saved plot: 2_univariate_BMI_distribution.png

Stats for MentHlth:
 count    229474.000000
mean          3.509866
std           7.717643
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          30.000000
Name: MentHlth, dtype: float64
Skewness: 2.5445155990241597, Kurtosis: 5.4353507353710455
Saved plot: 2_univariate_MentHlth_distribution.png

Stats for PhysHlth:
 count    229474.000000
mean          4.681219
std           9.050877
min           0.000000
25%           0.000000
50%           0.000000
75%           4.000000
max          30.000000
Name: PhysHlth, dtype: float64
Skewness: 2.0441257821599637, Kurtosis: 2.76150880926584
Saved plot: 2_univariate_Phy

In [34]:

target = 'Diabetes_binary'

# Continuous: Violin/Box + Tests
for col in continuous_features:
    sns.violinplot(x=df_viz[target], y=df[col], palette=color_palette, inner='quartile')
    plt.title(f'{col} by Diabetes Status')
    save_plot(f'3_bivariate_violin_{col}_vs_diabetes')
    
    group0 = df[df[target] == 0][col].dropna()
    group1 = df[df[target] == 1][col].dropna()
    test_type, p, effect = perform_stat_test(group0, group1, col)
    print(f"{test_type} for {col}: p-value = {p:.4e}, Cohen's d = {effect:.2f}")

# Categorical: Stacked Bars + Chi-square
for col in categorical_features:
    if col != target:
        crosstab = pd.crosstab(df_viz[col], df_viz[target], normalize='index') * 100
        crosstab.plot(kind='bar', stacked=True, colormap='coolwarm')
        plt.title(f'Diabetes Prevalence by {col}')
        plt.ylabel('Proportion (%)')
        plt.xticks(rotation=45, ha='right')
        save_plot(f'3_bivariate_stacked_{col}')
        
        table = pd.crosstab(df[col], df[target])
        chi2, p, _, _ = chi2_contingency(table)
        n = table.sum().sum()
        cramers_v = calc_cramers_v(chi2, n, table.shape[0], table.shape[1])
        print(f"Chi-square for {col}: p-value = {p:.4e}, Cramér's V = {cramers_v:.2f}")

# Odds Ratios
print("\nOdds Ratios for Binary Features:")
odds_ratios = {}
for col in [c for c in binary_cols if c != target]:
    table = pd.crosstab(df[col], df[target])
    res = sm.stats.Table2x2(table.to_numpy())
    or_val = res.oddsratio
    lower, upper = res.oddsratio_confint()
    print(f"{col}: OR = {or_val:.2f} (95% CI: {lower:.2f}-{upper:.2f})")

Saved plot: 3_bivariate_violin_BMI_vs_diabetes.png
Mann-Whitney for BMI: p-value = 0.0000e+00, Cohen's d = 0.59
Saved plot: 3_bivariate_violin_MentHlth_vs_diabetes.png
Mann-Whitney for MentHlth: p-value = 1.1019e-19, Cohen's d = 0.16
Saved plot: 3_bivariate_violin_PhysHlth_vs_diabetes.png
Mann-Whitney for PhysHlth: p-value = 0.0000e+00, Cohen's d = 0.47
Saved plot: 3_bivariate_stacked_GenHlth.png
Chi-square for GenHlth: p-value = 0.0000e+00, Cramér's V = 0.28
Saved plot: 3_bivariate_stacked_Age.png
Chi-square for Age: p-value = 0.0000e+00, Cramér's V = 0.19
Saved plot: 3_bivariate_stacked_Education.png
Chi-square for Education: p-value = 0.0000e+00, Cramér's V = 0.10
Saved plot: 3_bivariate_stacked_Income.png
Chi-square for Income: p-value = 0.0000e+00, Cramér's V = 0.14
Saved plot: 3_bivariate_stacked_HighBP.png
Chi-square for HighBP: p-value = 0.0000e+00, Cramér's V = 0.25
Saved plot: 3_bivariate_stacked_HighChol.png
Chi-square for HighChol: p-value = 0.0000e+00, Cramér's V = 0.19
Sa

In [35]:

# Correlation Heatmap
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr(method='spearman')
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Spearman Correlation Matrix')
save_plot('4_multivariate_correlation_heatmap')

# Pairplots (subset for performance)
sns.pairplot(df.sample(1000)[continuous_features + [target]], hue=target, palette=color_palette)
save_plot('4_multivariate_pairplot')

# VIF for Multicollinearity
print("\nVariance Inflation Factor (VIF):")
X_vif = df[numeric_cols].drop(target, axis=1, errors='ignore').astype(float)
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))

high_vif = vif_data[vif_data['VIF'] > 5]['feature'].tolist()
if high_vif:
    print(f"Dropping high VIF features: {high_vif}")
    X_vif_dropped = X_vif.drop(columns=high_vif)
    corr_dropped = X_vif_dropped.corr(method='spearman')
    sns.heatmap(corr_dropped, annot=False, cmap='coolwarm')
    plt.title('Correlation After Dropping High VIF')
    save_plot('4_multivariate_correlation_dropped')

# PCA
X_numeric = df[numeric_cols].drop(target, axis=1)
X_scaled = StandardScaler().fit_transform(X_numeric)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]
sns.scatterplot(data=df.sample(5000), x='pca1', y='pca2', hue=target, palette=color_palette, alpha=0.7)
plt.title('PCA Colored by Diabetes')
save_plot('4_multivariate_pca')

# t-SNE (on sample)
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_scaled[:5000])
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=df[target][:5000], palette=color_palette)
plt.title('t-SNE Colored by Diabetes')
save_plot('4_multivariate_tsne')

# Clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)
sns.scatterplot(data=df.sample(5000), x='pca1', y='pca2', hue='KMeans_Cluster', palette=color_palette)
plt.title('K-Means Clusters on PCA')
save_plot('4_multivariate_kmeans')

# Fixed DBSCAN on sample
sample_idx = np.random.choice(len(X_scaled), 50000, replace=False)
dbscan = DBSCAN(eps=1.5, min_samples=100)
clusters = dbscan.fit_predict(X_scaled[sample_idx])
df['DBSCAN_Cluster'] = -2  # Default for non-sampled
df.loc[df.index[sample_idx], 'DBSCAN_Cluster'] = clusters
unique_clusters = len(set(clusters[clusters != -1]))
print(f"DBSCAN found {unique_clusters} clusters in the sample (excluding noise).")


Saved plot: 4_multivariate_correlation_heatmap.png
Saved plot: 4_multivariate_pairplot.png

Variance Inflation Factor (VIF):
                 feature        VIF
0                 HighBP        inf
5                 Stroke        inf
15              DiffWalk        inf
1               HighChol        inf
6   HeartDiseaseorAttack        inf
16      NumComorbidities        inf
2              CholCheck  18.519049
11         AnyHealthcare  15.477796
3                    BMI  14.591856
9                Veggies   5.062478
7           PhysActivity   3.947097
8                 Fruits   2.763740
4                 Smoker   1.916126
14              PhysHlth   1.800025
13              MentHlth   1.407428
12           NoDocbcCost   1.185977
10     HvyAlcoholConsump   1.083131
Dropping high VIF features: ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'Veggies', 'AnyHealthcare', 'DiffWalk', 'NumComorbidities']
Saved plot: 4_multivariate_correlation_dropped.png
Saved plot:

In [36]:

bmi_bins = [0, 18.5, 25, 30, np.inf]
bmi_labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
df['BMI_Category'] = pd.cut(df['BMI'], bins=bmi_bins, labels=bmi_labels)

# Facet Plot
g = sns.catplot(data=df, x='BMI_Category', y=target, hue='Sex', col='Age', kind='bar', palette=color_palette, col_wrap=5, height=4)
g.set_axis_labels("BMI Category", "Diabetes Prevalence")
save_plot('5_stratification_bmi_sex_age')

# Prevalence and Risks by Group
groups = ['Age', 'BMI_Category', 'Sex', 'Smoker', 'HvyAlcoholConsump']
for group in groups:
    prev = df.groupby(group)[target].mean().sort_values(ascending=False)
    print(f"\nDiabetes Prevalence by {group}:\n{prev}")
    
# Interaction Example
formula = "Diabetes_binary ~ BMI * PhysActivity + Age"
interaction_model = sm_logit(formula, data=df).fit(disp=0)
print(interaction_model.summary())


Saved plot: 5_stratification_bmi_sex_age.png

Diabetes Prevalence by Age:
Age
70-74          0.231437
65-69          0.222837
75-79          0.219975
60-64          0.191421
80 or older    0.190578
55-59          0.155507
50-54          0.132866
45-49          0.100752
40-44          0.074715
35-39          0.051108
30-34          0.031328
25-29          0.019819
18-24          0.014154
Name: Diabetes_binary, dtype: float64

Diabetes Prevalence by BMI_Category:
BMI_Category
Obese          0.254424
Overweight     0.140258
Normal         0.072656
Underweight    0.055374
Name: Diabetes_binary, dtype: float64

Diabetes Prevalence by Sex:
Sex
Male      0.166258
Female    0.142524
Name: Diabetes_binary, dtype: float64

Diabetes Prevalence by Smoker:
Smoker
1    0.170485
0    0.137651
Name: Diabetes_binary, dtype: float64

Diabetes Prevalence by HvyAlcoholConsump:
HvyAlcoholConsump
0    0.158985
1    0.059642
Name: Diabetes_binary, dtype: float64
                           Logit Regression Re

In [37]:

comorbidity_cols = ['HeartDiseaseorAttack', 'Stroke', 'HighBP', 'HighChol']

# Co-occurrence Heatmap
cooccur = pd.crosstab(df[target], df['NumComorbidities'])
sns.heatmap(cooccur, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Comorbidity Count by Diabetes')
save_plot('6_comorbidity_heatmap')

# Odds Ratios / Relative Risks
for col in comorbidity_cols:
    table = pd.crosstab(df[col], df[target])
    risk_exposed = float((table.iloc[1,1] / table.iloc[1].sum().fillna(0).astype(float)) if table.iloc[1].sum() else 0)
    risk_unexposed = float((table.iloc[0,1] / table.iloc[0].sum().fillna(0).astype(float)) if table.iloc[0].sum() else 0)
    rr = risk_exposed / risk_unexposed if risk_unexposed > 0 else 0.0
    print(f"Relative Risk for {col}: {rr:.2f}")

# UpSet if available
if UpSet:
    upset_data = df[comorbidity_cols + [target]].set_index(comorbidity_cols)
    UpSet(upset_data, subset_size='count', show_percentages=True)

# Venn (for 2)
if venn2:
    set1 = set(df[df['HighBP'] == 1].index)
    set2 = set(df[df['HighChol'] == 1].index)
    intersection = len(set1 & set2)
    only1 = len(set1 - set2)
    only2 = len(set2 - set1)
    venn2(subsets=(only1, only2, intersection), set_labels=('HighBP', 'HighChol'))
    plt.title('Venn Diagram of HighBP and HighChol')
    save_plot('6_comorbidity_venn')

# Co-occurrence Matrix
comorb_matrix = pd.DataFrame(index=comorbidity_cols, columns=comorbidity_cols)
for i in comorb_matrix.index:
    for j in comorb_matrix.columns:
        if i != j:
            comorb_matrix.loc[i, j] = len(df[(df[i]==1) & (df[j]==1)])
sns.heatmap(comorb_matrix.astype(float), annot=True, cmap='Blues')
plt.title('Pairwise Comorbidity Co-occurrence')
save_plot('6_comorb_matrix')

# Attributable Risk
P_disease = df[target].mean()
for col in comorbidity_cols:
    P_exposed = df[col].mean()
    ar = risk_exposed - risk_unexposed
    par = P_exposed * ar / P_disease if P_disease > 0 else 0.0
    print(f"Attributable Risk for {col}: {ar:.2%}, PAR: {par:.2%}")


Saved plot: 6_comorbidity_heatmap.png


AttributeError: 'numpy.int64' object has no attribute 'fillna'

In [30]:
print("\n--- STEP 7: FEATURE IMPORTANCE & PREDICTIVE EXPLORATION ---")

# Prepare Data
genhlth_map_num = {'Excellent':1, 'Very Good':2, 'Good':3, 'Fair':4, 'Poor':5}
df['GenHlth_numeric'] = df['GenHlth'].map(genhlth_map_num)
df_encoded = pd.get_dummies(df.drop(columns=[target, 'KMeans_Cluster', 'DBSCAN_Cluster', 'pca1', 'pca2', 'NumComorbidities', 'BMI_Category', 'Age']), drop_first=True)
# Sanitize feature names for XGBoost
df_encoded.columns = [col.replace('<', 'lt_').replace('>', 'gt_').replace('[', '_').replace(']', '_').replace(' ', '_').replace(',', '_').replace('(', '_').replace(')', '_') for col in df_encoded.columns]
X = pd.concat([df_encoded, df['Age_numeric']], axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# RFECV
log_reg_rfe = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
rfecv = RFECV(estimator=log_reg_rfe, cv=StratifiedKFold(3), scoring='roc_auc', n_jobs=-1)
rfecv.fit(X_train_scaled, y_train)
X_train_rfe = rfecv.transform(X_train_scaled)
X_test_rfe = rfecv.transform(X_test_scaled)

# Models with CV
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=len(y)/sum(y))
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    if "Logistic" in name:
        scores = cross_val_score(model, X_train_rfe, y_train, cv=cv, scoring='roc_auc')
    else:
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f"CV AUC for {name}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    model.fit(X_train if "Logistic" not in name else X_train_rfe, y_train)
    y_pred_proba = model.predict_proba(X_test if "Logistic" not in name else X_test_rfe)[:, 1]
    
    PrecisionRecallDisplay.from_predictions(y_test, y_pred_proba)
    plt.title(f'PR Curve for {name}')
    save_plot(f'7_pr_curve_{name}')

# SHAP for XGBoost
explainer = shap.TreeExplainer(models['XGBoost'])
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)  # Convert to DataFrame for sampling
typing.cast(pd.DataFrame, X_test_df)  # Type hint

shap_values = explainer(X_test_df.sample(2000))
shap.summary_plot(shap_values, X_test_df.sample(2000))

# Enhanced SHAP
shap.dependence_plot('BMI', shap_values.values, X_test.sample(2000), interaction_index='Age_numeric')
save_plot('7_shap_dependence_bmi_age')

# PDP for Top Features
top_features = ['BMI', 'Age_numeric', 'GenHlth_numeric', 'HighBP']
PartialDependenceDisplay.from_estimator(models['Random Forest'], X_train, top_features)
save_plot('7_pdp')

# Enhanced PDP
PartialDependenceDisplay.from_estimator(models['Random Forest'], X_train, [('BMI', 'Age_numeric')])
save_plot('7_pdp_interaction')

print("Medical Insight: Top predictors like BMI/GenHlth match literature; high AUC suggests predictive utility for risk screening.")



--- STEP 7: FEATURE IMPORTANCE & PREDICTIVE EXPLORATION ---


KeyError: 'Age_numeric'